# Hub-Spoke Architecture: Centralized Azure Firewall

## Customer Recommendation

**Yes — Azure Firewall should be deployed to a central Hub VNet** that routes traffic for all spoke workloads.
This is Microsoft's recommended enterprise pattern and aligns with Azure Landing Zone best practices.

---

## Why Hub-Spoke with Centralized Firewall?

### Current MVP (Single VNet — OK for Demo)

```mermaid
graph TB
    subgraph SingleVNet["Single VNet (10.0.0.0/16)"]
        FWSub["🔥 Firewall Subnet"]
        ACASub["📦 ACA Subnet<br/>(ActiveMQ)"]
    end

    style SingleVNet fill:#fff3cd,stroke:#ffc107,stroke-width:2px
    style FWSub fill:#f8d7da,stroke:#dc3545
    style ACASub fill:#d1ecf1,stroke:#0dcaf0
```

> ⚠️ **Problem**: Firewall is coupled to ONE workload VNet. Can't share across projects.

### Recommended: Hub-Spoke Topology

```mermaid
graph TB
    Internet["🌐 INTERNET"] --> FW

    subgraph Hub["Hub VNet (10.0.0.0/16)"]
        FW["🔥 Azure Firewall<br/>PIP: 20.x.x.x<br/>DNAT + SNAT + Network Rules"]
        GW["🔌 Gateway Subnet<br/>(VPN/ExpressRoute)"]
    end

    FW -- "VNet Peering" --> Spoke1
    FW -- "VNet Peering" --> Spoke2
    FW -- "VNet Peering" --> Spoke3

    subgraph Spoke1["Spoke 1 (10.1.0.0/16)"]
        AMQ["📦 ActiveMQ<br/>(ACA)"]
        UDR1["🔀 UDR → Hub FW"]
    end

    subgraph Spoke2["Spoke 2 (10.2.0.0/16)"]
        WebApp["🌐 Web App<br/>(ACA)"]
        UDR2["🔀 UDR → Hub FW"]
    end

    subgraph Spoke3["Spoke N (10.N.0.0/16)"]
        Future["🚀 Future<br/>Workload"]
        UDR3["🔀 UDR → Hub FW"]
    end

    style Hub fill:#e8f5e9,stroke:#4caf50,stroke-width:3px
    style Spoke1 fill:#e3f2fd,stroke:#2196f3,stroke-width:2px
    style Spoke2 fill:#fce4ec,stroke:#e91e63,stroke-width:2px
    style Spoke3 fill:#f3e5f5,stroke:#9c27b0,stroke-width:2px
    style FW fill:#ff9800,color:#fff,stroke:#e65100,stroke-width:2px
    style Internet fill:#263238,color:#fff,stroke:#263238
```

> ✅ ONE Firewall secures ALL spoke workloads | ✅ Centralized egress | ✅ Spokes are isolated

## Single VNet vs Hub-Spoke Comparison

| Criteria | Single VNet (Current MVP) | Hub-Spoke (Recommended) |
|----------|--------------------------|-------------------------|
| **Complexity** | Simple — 1 VNet, 1 RG | More setup — Hub + Spoke VNets + Peering |
| **Cost** | 1 Firewall for 1 workload | 1 Firewall for N workloads (amortized) |
| **Scalability** | Add subnets to same VNet | Add new Spoke VNets (no Hub changes) |
| **Isolation** | All workloads share address space | Each spoke is fully isolated |
| **Blast Radius** | Compromise = access to everything | Compromise of Spoke 1 ≠ Spoke 2 |
| **IP Management** | One CIDR block, can get crowded | Each spoke has its own CIDR |
| **Egress IP** | Firewall PIP = egress for that VNet | Same Firewall PIP = egress for ALL spokes |
| **Team Ownership** | One team manages everything | NetOps owns Hub, App teams own Spokes |
| **Compliance** | Harder to audit per-workload | Per-spoke audit + central firewall logs |
| **Use Case** | Dev/demo, single workload | **Production, multi-workload, enterprise** |

### When to Use Which?

| Scenario | Recommendation |
|----------|---------------|
| Quick demo / POC | Single VNet (simpler) |
| 1 workload, small team | Single VNet is fine |
| 2+ workloads, shared egress IP | **Hub-Spoke** |
| Enterprise / compliance requirements | **Hub-Spoke** |
| Multi-team organization | **Hub-Spoke** (separation of duties) |
| Azure Landing Zone adoption | **Hub-Spoke** (it's the default pattern) |

## How Traffic Flows in Hub-Spoke

### Inbound (Client → ActiveMQ)

```mermaid
sequenceDiagram
    participant Client as 🧑‍💻 External Client<br/>(Whitelisted IP)
    participant FW as 🔥 Azure Firewall<br/>Hub VNet
    participant Peer as 🔗 VNet Peering<br/>Hub ↔ Spoke
    participant AMQ as 📦 ActiveMQ<br/>Spoke VNet (ACA)

    Client->>FW: TCP SYN → FW PIP:61616
    FW->>FW: Check DNAT whitelist
    Note over FW: ✅ Source IP matched!
    FW->>Peer: DNAT → 10.1.x.x:61616
    Peer->>AMQ: Forward translated packet
    AMQ-->>Client: TCP SYN-ACK (via Firewall SNAT)
    Client->>AMQ: OpenWire / STOMP messages ✅
```

### Outbound (ActiveMQ → Internet)

```mermaid
sequenceDiagram
    participant AMQ as 📦 ActiveMQ<br/>Spoke VNet
    participant UDR as 🔀 UDR (Route Table)<br/>0.0.0.0/0 → FW
    participant FW as 🔥 Azure Firewall<br/>Hub VNet
    participant Internet as 🌐 Internet

    AMQ->>UDR: Outbound request (e.g., container pull)
    UDR->>FW: Route via VNet Peering → Hub FW
    FW->>FW: Check Network Rules
    Note over FW: ✅ Allow DNS (53)<br/>✅ Allow HTTPS (443)<br/>❌ Deny all else
    FW->>Internet: SNAT → Firewall PIP (20.x.x.x)
    Internet-->>FW: Response
    FW-->>AMQ: Forward back via Peering
```

### Spoke-to-Spoke (Optional, via Firewall)

```mermaid
graph LR
    Spoke1["📦 Spoke 1<br/>ActiveMQ<br/>10.1.0.0/16"] -- "UDR: 10.2.0.0/16<br/>→ Hub FW" --> FW
    FW["🔥 Azure Firewall<br/>(Inspects + Logs)"] -- "Network Rule: Allow<br/>10.1 → 10.2" --> Spoke2["🌐 Spoke 2<br/>Web App<br/>10.2.0.0/16"]

    style FW fill:#ff9800,color:#fff,stroke:#e65100,stroke-width:2px
    style Spoke1 fill:#e3f2fd,stroke:#2196f3,stroke-width:2px
    style Spoke2 fill:#fce4ec,stroke:#e91e63,stroke-width:2px
```

> ✅ All spoke-to-spoke traffic is inspected and logged by the centralized Firewall

## Azure Firewall Egress IP (SNAT) in Hub-Spoke

This is the key value proposition for centralized Firewall:

| Configuration | Behavior | Use Case |
|--------------|----------|----------|
| **1 Firewall PIP** | ALL spokes egress via that single IP | Simple, predictable |
| **Multiple PIPs** | Azure picks any PIP for SNAT | More SNAT port capacity |
| **Public IP Prefix** | Contiguous IP range for SNAT | Partners whitelist a CIDR block |

### Why This Matters for Customers

```mermaid
graph TB
    Spoke1["📦 Spoke 1<br/>ActiveMQ<br/>10.1.x.x"] --> FW
    Spoke2["🌐 Spoke 2<br/>Web API<br/>10.2.x.x"] --> FW
    Spoke3["⚙️ Spoke 3<br/>Batch Jobs<br/>10.3.x.x"] --> FW

    FW["🔥 Azure Firewall<br/>SNAT → 20.119.45.x"] --> Partner["🤝 Partner API<br/>Whitelist:<br/>20.119.45.123 ✅"]

    style FW fill:#ff9800,color:#fff,stroke:#e65100,stroke-width:2px
    style Partner fill:#e8f5e9,stroke:#4caf50,stroke-width:2px
    style Spoke1 fill:#e3f2fd,stroke:#2196f3
    style Spoke2 fill:#fce4ec,stroke:#e91e63
    style Spoke3 fill:#f3e5f5,stroke:#9c27b0
```

> **Without centralized Firewall**: Each workload needs its own NAT Gateway or PIP → partners whitelist N different IPs.
> **With centralized Firewall**: Partners whitelist ONE IP for ALL workloads.

## Hub-Spoke: Key Configuration Components

### 1. VNet Peering Settings

Both directions must be configured correctly:

| Setting | Hub → Spoke | Spoke → Hub |
|---------|-------------|-------------|
| Allow forwarded traffic | ✅ Yes | ✅ Yes |
| Allow gateway transit | ✅ Yes (if VPN/ER) | ❌ No |
| Use remote gateways | ❌ No | ✅ Yes (if VPN/ER) |
| Allow virtual network access | ✅ Yes | ✅ Yes |

### 2. Route Table (UDR) on Spoke Subnets

| Destination | Next Hop Type | Next Hop IP |
|-------------|--------------|-------------|
| 0.0.0.0/0 | Virtual Appliance | Hub Firewall Private IP (e.g., 10.0.1.4) |
| 10.2.0.0/16 (other spoke) | Virtual Appliance | Hub Firewall Private IP |

### 3. Firewall Rules

| Rule Type | Purpose | Example |
|-----------|---------|--------|
| DNAT | Inbound to spoke workloads | FW_PIP:61616 → 10.1.0.x:61616 |
| Network | Outbound from spokes | Allow 10.1.0.0/16 → Internet:443 |
| Network | Spoke-to-spoke | Allow 10.1.0.0/16 → 10.2.0.0/16:* |
| Application | FQDN filtering | Allow *.docker.io, *.azure.com |

## Bicep: Hub-Spoke Architecture

The following cells show the key Bicep modules for a hub-spoke deployment.
Full IaC is in the `infra/hub-spoke/` directory.

In [ ]:
# =============================================================================
# Hub-Spoke Architecture - Key Bicep Snippets (Reference)
# =============================================================================
# These show the critical configuration differences vs single-VNet
# =============================================================================

hub_vnet_bicep = """
// ===== HUB VNET =====
// Central VNet hosting the Azure Firewall
// Address space: 10.0.0.0/16

resource hubVnet 'Microsoft.Network/virtualNetworks@2023-11-01' = {
  name: 'hub-vnet'
  location: location
  properties: {
    addressSpace: {
      addressPrefixes: ['10.0.0.0/16']
    }
    subnets: [
      {
        name: 'AzureFirewallSubnet'        // Required name
        properties: { addressPrefix: '10.0.1.0/24' }
      }
      {
        name: 'AzureFirewallManagementSubnet'  // Required for Basic SKU
        properties: { addressPrefix: '10.0.2.0/24' }
      }
      {
        name: 'GatewaySubnet'               // For VPN/ExpressRoute
        properties: { addressPrefix: '10.0.3.0/24' }
      }
    ]
  }
}
"""

spoke_vnet_bicep = """
// ===== SPOKE VNET (per workload) =====
// Each spoke has its own VNet, isolated from other spokes
// Address space: 10.1.0.0/16 (Spoke 1), 10.2.0.0/16 (Spoke 2), etc.

resource spokeVnet 'Microsoft.Network/virtualNetworks@2023-11-01' = {
  name: 'spoke-activemq-vnet'
  location: location
  properties: {
    addressSpace: {
      addressPrefixes: ['10.1.0.0/16']
    }
    subnets: [
      {
        name: 'aca-subnet'
        properties: {
          addressPrefix: '10.1.0.0/20'
          routeTable: { id: spokeRouteTable.id }  // UDR → Hub Firewall
          delegations: [{
            name: 'aca-delegation'
            properties: { serviceName: 'Microsoft.App/environments' }
          }]
        }
      }
    ]
  }
}
"""

peering_bicep = """
// ===== VNET PEERING =====
// Bi-directional peering between Hub and Spoke

// Hub → Spoke peering
resource hubToSpoke 'Microsoft.Network/virtualNetworks/virtualNetworkPeerings@2023-11-01' = {
  parent: hubVnet
  name: 'hub-to-spoke-activemq'
  properties: {
    remoteVirtualNetwork: { id: spokeVnet.id }
    allowVirtualNetworkAccess: true
    allowForwardedTraffic: true       // ← Critical: allow Firewall to forward
    allowGatewayTransit: true          // If using VPN/ExpressRoute in Hub
  }
}

// Spoke → Hub peering
resource spokeToHub 'Microsoft.Network/virtualNetworks/virtualNetworkPeerings@2023-11-01' = {
  parent: spokeVnet
  name: 'spoke-activemq-to-hub'
  properties: {
    remoteVirtualNetwork: { id: hubVnet.id }
    allowVirtualNetworkAccess: true
    allowForwardedTraffic: true       // ← Critical: allow traffic via Firewall
    useRemoteGateways: false           // Set true if Hub has VPN/ER Gateway
  }
}
"""

udr_bicep = """
// ===== ROUTE TABLE (on Spoke subnets) =====
// Forces all egress through Hub Firewall

resource spokeRouteTable 'Microsoft.Network/routeTables@2023-11-01' = {
  name: 'spoke-activemq-rt'
  location: location
  properties: {
    disableBgpRoutePropagation: true   // Prevent BGP from overriding UDR
    routes: [
      {
        name: 'to-hub-firewall'
        properties: {
          addressPrefix: '0.0.0.0/0'            // ALL traffic
          nextHopType: 'VirtualAppliance'
          nextHopIpAddress: '10.0.1.4'           // Hub Firewall private IP
        }
      }
      {
        name: 'to-spoke2-via-firewall'           // Spoke-to-spoke via FW
        properties: {
          addressPrefix: '10.2.0.0/16'
          nextHopType: 'VirtualAppliance'
          nextHopIpAddress: '10.0.1.4'
        }
      }
    ]
  }
}
"""

print("📋 Key Hub-Spoke Bicep Components:")
print("="*60)
components = [
    ("Hub VNet",    "Central VNet with Firewall + Gateway subnets"),
    ("Spoke VNet",  "Workload VNet with ACA subnet + UDR"),
    ("VNet Peering", "Bi-directional with allowForwardedTraffic=true"),
    ("Route Table",  "UDR: 0.0.0.0/0 → Hub Firewall Private IP"),
    ("Firewall",     "Same as before, but in Hub VNet (shared)"),
]
for name, desc in components:
    print(f"  ✅ {name:<18} → {desc}")

print(f"\n💡 Key difference from single VNet:")
print(f"   • VNet Peering connects Hub ↔ Spoke")
print(f"   • allowForwardedTraffic = true (CRITICAL)")
print(f"   • UDR nextHopIpAddress points to Hub Firewall's PRIVATE IP")
print(f"   • Firewall is shared across all spokes (cost amortized)")

## Cost Impact: Hub-Spoke vs Single VNet

Hub-Spoke adds minimal extra cost — the Firewall is the same, you just add VNet Peering.

In [ ]:
# =============================================================================
# Cost Comparison: Single VNet vs Hub-Spoke
# =============================================================================

print("=" * 85)
print("  COST COMPARISON: Single VNet vs Hub-Spoke")
print("  Region: East US 2 | Azure Firewall Basic SKU")
print("=" * 85)

# Single VNet costs (same as before)
single_vnet = {
    "Azure Firewall Basic":      288.35,
    "FW Data Processing (100GB)": 1.60,
    "Public IPs (x2)":            7.30,
    "Container Apps (1vCPU/2GiB)": 21.90,
    "Log Analytics (5GB)":        13.80,
    "VNet":                        0.00,
}

# Hub-Spoke adds VNet Peering cost
hub_spoke = {
    **single_vnet,
    "VNet Peering (ingress)": 0.01 * 100,   # $0.01/GB, ~100 GB
    "VNet Peering (egress)":  0.01 * 100,   # $0.01/GB, ~100 GB
}
# Hub VNet is also free
hub_spoke["Hub VNet"] = 0.00

print(f"\n{'Component':<35} {'Single VNet':>14} {'Hub-Spoke':>14}")
print(f"{'-'*35} {'-'*14} {'-'*14}")

total_single = 0
total_hub = 0

all_keys = list(dict.fromkeys(list(single_vnet.keys()) + list(hub_spoke.keys())))
for key in all_keys:
    sv = single_vnet.get(key, None)
    hs = hub_spoke.get(key, None)
    sv_str = f"${sv:>10.2f}" if sv is not None else f"{'—':>11}"
    hs_str = f"${hs:>10.2f}" if hs is not None else f"{'—':>11}"
    if sv: total_single += sv
    if hs: total_hub += hs
    new = " ← NEW" if key not in single_vnet else ""
    print(f"  {key:<33} {sv_str:>14} {hs_str:>14}{new}")

print(f"{'-'*35} {'-'*14} {'-'*14}")
print(f"  {'TOTAL':<33} ${total_single:>10.2f}/mo ${total_hub:>10.2f}/mo")
diff = total_hub - total_single
print(f"  {'Delta':<33} {'':>14} ${diff:>+10.2f}/mo")

print(f"\n💡 Key Takeaways:")
print(f"  • Hub-Spoke adds only ~${diff:.2f}/mo for VNet Peering data transfer")
print(f"  • VNet Peering: $0.01/GB in each direction (very cheap)")
print(f"  • The Firewall cost is the SAME — you just share it across spokes")
print(f"  • Adding Spoke 2 costs $0/mo extra for Firewall (already running)")
print(f"  • Only additional cost per spoke: VNet Peering data transfer")

print(f"\n📊 With 3 Spokes sharing 1 Firewall:")
fw_per_spoke = 288.35 / 3
print(f"  • Firewall cost per spoke: ${fw_per_spoke:.2f}/mo (vs ${288.35:.2f} each if isolated)")
print(f"  • Savings vs 3 separate Firewalls: ${288.35 * 2:.2f}/mo")

## Migration Path: Single VNet → Hub-Spoke

If you start with the Single VNet MVP and need to migrate to Hub-Spoke later:

| Step | Action | Downtime? |
|------|--------|----------|
| 1 | Deploy Hub VNet with Firewall | No (new resources) |
| 2 | Deploy new Spoke VNet with ACA | No (new resources) |
| 3 | Set up VNet Peering (Hub ↔ Spoke) | No |
| 4 | Configure UDR on Spoke subnet | No |
| 5 | Deploy ActiveMQ to new Spoke ACA | No |
| 6 | Update Firewall DNAT to point to new Spoke | Brief DNS/IP cutover |
| 7 | Decommission old single-VNet setup | No |

### Recommendation for Your Customer

| Phase | Topology | Justification |
|-------|----------|---------------|
| **POC / Demo** | Single VNet (current) | Fast to deploy, easy to explain |
| **Pilot / UAT** | Hub-Spoke | Matches prod architecture, validates peering |
| **Production** | Hub-Spoke | Mandatory for enterprise, compliance, multi-workload |

## Summary: Customer Recommendation

### ✅ YES — Deploy Firewall to a Central Hub VNet

```mermaid
graph TB
    subgraph Recommendation["✅ Production Architecture"]
        direction TB
        Internet["🌐 Internet"] --> FW
        FW["🔥 Azure Firewall<br/>Hub VNet<br/>Basic SKU ~$288/mo<br/>Shared across ALL spokes"]
        FW -- "DNAT" --> S1["📦 Spoke: ActiveMQ"]
        FW -- "DNAT" --> S2["🌐 Spoke: Web Apps"]
        FW -- "DNAT" --> S3["⚙️ Spoke: APIs"]
        S1 -- "SNAT via FW" --> FW
        S2 -- "SNAT via FW" --> FW
        S3 -- "SNAT via FW" --> FW
    end

    style Recommendation fill:#e8f5e9,stroke:#4caf50,stroke-width:3px
    style FW fill:#ff9800,color:#fff,stroke:#e65100,stroke-width:2px
```

**Why Hub-Spoke?**

| # | Reason | Detail |
|---|--------|--------|
| 1 | **Single Egress IP** | All spokes share the same Firewall PIP for SNAT |
| 2 | **Cost Efficiency** | One Firewall ($288/mo) shared across all workloads |
| 3 | **Centralized Security** | One place for DNAT, network rules, threat intel, logs |
| 4 | **Workload Isolation** | Each spoke VNet is isolated from others |
| 5 | **Scalability** | New workload = new Spoke VNet + peer to Hub |
| 6 | **Azure Landing Zone** | Microsoft-recommended enterprise architecture (CAF) |
| 7 | **Separation of Duties** | Network team owns Hub, app teams own Spokes |

### SNAT Reminder

| Firewall Config | Egress Behavior |
|----------------|------------------|
| 1 Public IP | All spokes egress via that IP (deterministic) |
| Multiple PIPs | Any PIP can be used for SNAT |
| IP Prefix | Contiguous range — easy to whitelist as CIDR |

### Next Steps
1. Use the current **Single VNet MVP** for your demo (Notebook 01 + 02)
2. When ready for production, deploy the **Hub-Spoke Bicep** from `infra/hub-spoke/`
3. Peer your workload spokes to the Hub
4. All traffic flows through the centralized Firewall